# Deploy mistralai/Voxtral-Mini-4B-Realtime-2602 with vLLM on SageMaker AI

This notebook demonstrates how to deploy mistralai/Voxtral-Mini-4B-Realtime-2602 model using vLLM Realtime API with WebSockets on Amazon SageMaker AI Bi-Directional streamin endpoint for high-performance inference.

## What We'll Do:
1. **Setup Environment**: Install dependencies and configure AWS
2. **Build Custom Container**: Create vLLM container with mistralai/Voxtral-Mini-4B-Realtime-2602
3. **Deploy to SageMaker**: Create endpoint for inference
4. **Test Deployment**: Validate bi-direction streaming inference capabilities

## Prerequisites:
- mistralai/Voxtral-Mini-4B-Realtime-2602 model from huggingFace
- AWS credentials configured
- Docker environment available

## 1. Environment Setup

In [ ]:
%pip install -r requirements.txt

In [119]:
import boto3
import os
from sagemaker.train.configs import InputData
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage
from sagemaker.core.helper.session_helper import Session

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
s3_client = boto3.client("s3")

## Install Docker

In [ ]:
%%bash

# see https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository
sudo apt-get update
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

## Currently only Docker version 20.10.X is supported in Studio: see https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html
# pick the latest patch from:
# apt-cache madison docker-ce | awk '{ print $3 }' | grep -i 20.10
VERSION_STRING=5:20.10.24~3-0~ubuntu-jammy
sudo apt-get install docker-ce-cli=$VERSION_STRING docker-compose-plugin -y

# validate the Docker Client is able to access Docker Server at [unix:///docker/proxy.sock]
docker version

## 4. Create Docker Artifacts for vLLM Deployment

In [ ]:
# Create docker artifacts directory
!mkdir -p voxtral-mini-4B-docker-artifacts

print("Docker artifacts directory created")

In [ ]:
%%writefile voxtral-mini-4B-docker-artifacts/Dockerfile
FROM public.ecr.aws/deep-learning-containers/vllm:0.17.0-gpu-py312-cu129-ubuntu22.04-sagemaker-v1.1-soci

# Set a docker label to advertise bidirectional streaming support on the container
LABEL com.amazonaws.sagemaker.capabilities.bidirectional-streaming=true

# Set working directory
WORKDIR /opt/ml/code

# Install dependencies audio model dependencies
COPY requirements.txt .
RUN pip install --upgrade --no-cache-dir -r requirements.txt

# Application code bridge: routes /invocations-bidirectional-stream → /v1/realtime
COPY app.py .

COPY sagemaker-entrypoint.sh entrypoint.sh
RUN chmod +x entrypoint.sh

ENTRYPOINT ["./entrypoint.sh"]

HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
  CMD curl -f http://localhost:$PORT/ping || exit 1

In [ ]:
%%writefile voxtral-mini-4B-docker-artifacts/sagemaker-entrypoint.sh
#!/bin/bash
# Starts vLLM on :8081 (internal) and the WebSocket bridge on :8080 (SageMaker-facing).
# The bridge routes /invocations-bidirectional-stream → /v1/realtime.

PREFIX="SM_VLLM_"
ARG_PREFIX="--"

# ── Bridge configuration ──
VLLM_INTERNAL_PORT="${VLLM_INTERNAL_PORT:-8081}"
BRIDGE_PORT=8080
BRIDGE_SCRIPT=/opt/ml/code/app.py

# Build vLLM args from SM_VLLM_ environment variables
ARGS=(--port "${VLLM_INTERNAL_PORT}")

# Loop through all environment variables
while IFS='=' read -r key value; do
    arg_name=$(echo "${key#"${PREFIX}"}" | tr '[:upper:]' '[:lower:]' | tr '_' '-')
    ARGS+=("${ARG_PREFIX}${arg_name}")
    [[ -n "$value" ]] && ARGS+=("$value")
done < <(env | grep "^${PREFIX}")


echo "============================================"
echo "  vLLM args: ${ARGS[*]}"
echo "  vLLM internal port: ${VLLM_INTERNAL_PORT}"
echo "  Bridge port: ${BRIDGE_PORT}"
echo "============================================"

# Signal handling
cleanup() {
    echo "Shutting down…"
    kill $VLLM_PID $BRIDGE_PID 2>/dev/null
    wait
}
trap cleanup SIGTERM SIGINT

# 1. Start vLLM on internal port (background)
python3 -m vllm.entrypoints.openai.api_server "${ARGS[@]}" &
VLLM_PID=$!

# 2. Wait for vLLM to be ready
echo "Waiting for vLLM on port ${VLLM_INTERNAL_PORT}..."
MAX_RETRIES=120
RETRY_COUNT=0
until curl -sf "http://localhost:${VLLM_INTERNAL_PORT}/health" > /dev/null 2>&1; do
    RETRY_COUNT=$((RETRY_COUNT + 1))
    if [ $RETRY_COUNT -ge $MAX_RETRIES ]; then
        echo "ERROR: vLLM failed to start after ${MAX_RETRIES} retries"
        kill $VLLM_PID 2>/dev/null
        exit 1
    fi
    # Check if vLLM process is still alive
    if ! kill -0 $VLLM_PID 2>/dev/null; then
        echo "ERROR: vLLM process died"
        exit 1
    fi
    sleep 2
done
echo "✓ vLLM ready on port ${VLLM_INTERNAL_PORT}"

# 3. Start the WebSocket bridge on port 8080 (SageMaker-facing)
echo "Starting WebSocket bridge on port ${BRIDGE_PORT}..."
export VLLM_INTERNAL_PORT
export BRIDGE_PORT
python3 "${BRIDGE_SCRIPT}" &
BRIDGE_PID=$!

echo "✓ Bridge ready on port ${BRIDGE_PORT}"

# 4. Wait for either process to exit — if one dies, shut down both
wait -n $VLLM_PID $BRIDGE_PID
EXIT_CODE=$?

cleanup
exit $EXIT_CODE

In [ ]:
%%writefile voxtral-mini-4B-docker-artifacts/build_and_push.sh
#!/bin/bash

# Build and push Docker image to ECR
REPO_NAME=$1
VERSION_TAG=$2
REGION=$3
ACCOUNT_ID=$4

# Create ECR repository if it doesn't exist
aws ecr describe-repositories --repository-names $REPO_NAME --region $REGION || \
aws ecr create-repository --repository-name $REPO_NAME --region $REGION

# Get ECR login token
aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com

# Build Docker image
docker build --network sagemaker -t $REPO_NAME:$VERSION_TAG .

# Tag image for ECR
docker tag $REPO_NAME:$VERSION_TAG $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG

# Push to ECR
docker push $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG

echo "Image pushed to: $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPO_NAME:$VERSION_TAG"

In [ ]:
# Make scripts executable
!chmod +x voxtral-mini-4B-docker-artifacts/sagemaker-entrypoint.sh
!chmod +x voxtral-mini-4B-docker-artifacts/build_and_push.sh

print("Docker scripts created and made executable")

## 5. Build and Push Docker Image

In [ ]:
account_id="xxxxxxxxxxx" # Your AWS account ID
region=sess.boto_region_name

In [ ]:
# Configuration for Docker image
REPO_NAME = "voxtral-mini-4b-realtime-2602-vllm"
VERSION_TAG = "latest"

print(f"Building Docker image: {REPO_NAME}:{VERSION_TAG}")
print(f"Target ECR: {account_id}.dkr.ecr.{region}.amazonaws.com/{REPO_NAME}:{VERSION_TAG}")

In [ ]:
%%bash -s {REPO_NAME} {VERSION_TAG} {region} {account_id}

REPO_NAME=$1
VERSION_TAG=$2
REGION=$3
ACCOUNT_ID=$4

echo "Building and pushing Docker image..."
cd voxtral-mini-4B-docker-artifacts && bash build_and_push.sh $REPO_NAME $VERSION_TAG $REGION $ACCOUNT_ID

In [ ]:
# Get the image URI
inference_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{REPO_NAME}:{VERSION_TAG}"
print(f"✅ Docker image URI: {inference_image}")

## Download the model from Hugging Face and upload the model artifacts on Amazon S3
If you are deploying a model hosted on the HuggingFace Hub, you must specify the `option.model_id=<hf_hub_model_id>` configuration. When using a model directly from the hub, we recommend you also specify the model revision (commit hash or branch) via `option.revision=<commit hash/branch>`. 

Since model artifacts are downloaded at runtime from the Hub, using a specific revision ensures you are using a model compatible with package versions in the runtime environment. Open Source model artifacts on the hub are subject to change at any time. These changes may cause issues when instantiating the model (updated model artifacts may require a newer version of a dependency than what is bundled in the container). If a model provides custom model (modeling.py) and/or custom tokenizer (tokenizer.py) files, you need to specify option.trust_remote_code=true to load and use the model.

In this example, we will demonstrate how to download your copy of the model from huggingface and upload it to an s3 location in your AWS account, then deploy the model with the downloaded model artifacts to an endpoint.  

**Best Practices**:
>
> **Store Models in Your Own S3 Bucket**
For production use-cases, always download and store model files in your own S3 bucket to ensure validated artifacts. This provides verified provenance, improved access control, consistent availability, protection against upstream changes, and compliance with organizational security protocols.
>

In [ ]:
HF_MODEL_ID = "mistralai/Voxtral-Mini-4B-Realtime-2602"

base_name = HF_MODEL_ID.split('/')[-1].replace('.', '-').lower()
model_lineage = HF_MODEL_ID.split("/")[0]
base_name

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sagemaker
import jinja2

# - This will download the model into the current directory where ever the jupyter notebook is running
local_model_path = Path(".")
local_model_path.mkdir(exist_ok=True)
model_name = HF_MODEL_ID
# Only download pytorch checkpoint files
allow_patterns = ["*.json", "*.safetensors", "*.bin", "*.txt"]

# - Leverage the snapshot library to donload the model since the model is stored in repository using LFS
model_download_path = snapshot_download(
    repo_id=model_name,
    cache_dir=local_model_path,
    allow_patterns=allow_patterns
)

### 6. Upload model files to S3
SageMaker AI allows us to provide [uncompressed](https://docs.aws.amazon.com/sagemaker/latest/dg/large-model-inference-uncompressed.html) files. Thus, we directly upload the folder that contains model files to s3
> **Note**: The default SageMaker bucket follows the naming pattern: `sagemaker-{region}-{account-id}`

In [13]:
s3_model_prefix = (
    "hf-large-models/Voxtral_Mini_4B_Realtime_2602"  # folder within bucket where model artifact will go
)

In [ ]:
model_artifact = sess.upload_data(path=model_download_path, key_prefix=s3_model_prefix)
print(f"Model uploaded to --- > {model_artifact}")

## 7. Configure Model Environment for the Model

In [101]:
instance_count = 1
instance_type = "ml.g6.4xlarge"
number_of_gpu = 1
health_check_timeout = 700

In [ ]:
# Environment variables for vLLM
vllm_env = {
    # Model configuration
    "SM_VLLM_MODEL": "/opt/ml/model", # SageMaker will automatically download the model at this location
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    # Performance configuration
    "SM_VLLM_MAX_MODEL_LEN": "45000",  # Context length To live-record a 1h meeting, you need to set --max-model-len >= 3600 / 0.8 = 45000
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9", # Conservative
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_MAX_NUM_SEQS": "8",     # Batch size
    "SM_VLLM_COMPILATION_CONFIG": '{"cudagraph_mode": "PIECEWISE"}',
    "SM_VLLM_DTYPE": "auto", # Precision
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

## 8. Deploy Model to SageMaker Endpoint

In [ ]:
import time
# Model and endpoint configuration
model_name = f"Voxtral-Mini-4B-Realtime-2602-{int(time.time())}"
endpoint_config_name=f"Voxtral-Mini-4B-Realtime-2602-config-{int(time.time())}"
endpoint_name = f"Voxtral-Mini-4B-Realtime-2602-endpoint-{int(time.time())}"

print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import ContainerDefinition, ModelDataSource, S3ModelDataSource, ModelAccessConfig, Tag
from sagemaker.core.helper.session_helper import get_execution_role

role = get_execution_role()

voxtral_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=f"{model_artifact}/",
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=vllm_env
    ),
    execution_role_arn=role,
)

pprint(voxtral_model)

### Create Endpoint Configuration

Define inference configuration

In [ ]:
import random
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config=EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_variant_weight=1.0,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ]
)

### Create Endpoint

A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

## 9. Test the Deployed Model

### Bidirectional Streaming - Realtime Audio Transcription

Test the vLLM Realtime API over SageMaker's bidirectional streaming transport. SageMaker uses HTTP/2 event streams on port 8443, which it bridges to WebSocket on the container's `/invocations-bidirectional-stream` endpoint (remapped from vLLM's `/v1/realtime`).

**Audio Format**

Audio must be sent as base64-encoded PCM16 audio at 16kHz sample rate, mono channel.

**[Protocol Overview](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/?h=realtime+api#realtime-api):**
1. Cient open bidirectional stream to the SageMaker endpoint
2. Server sends `session.created` event
3. Client sends `session.update` with the model path
4. Client sends `input_audio_buffer.commit` to signal readiness
5. Client streams base64 PCM16 chunks via `input_audio_buffer.append`
6. Server sends `transcription.delta` events with incremental text
7. Server sends `transcription.done` with final text + usage
8. Repeat from step 5 for next utterance
9. Optionally, client sends final `input_audio_buffer.commit` with `"final": true` to signal audio input is finished. Useful when streaming audio files

In [ ]:
!python sagemaker_bidi_client.py <your endpoint name> ./audio.wav

## 10. Cleanup (Optional)

In [ ]:
# Uncomment to delete the endpoint when done testing
# WARNING: This will delete your deployed model endpoint!

# print("⚠️ Deleting endpoint to avoid charges...")
# predictor.delete_endpoint()
# print(f"✅ Endpoint {endpoint_name} deleted")

print("💡 To delete the endpoint later, uncomment the code above or use:")
print(f"aws sagemaker delete-endpoint --endpoint-name {endpoint_name}")